Para agente 2

In [1]:
!pip install spacy

# Descargar modelo español
!python -m spacy download en_core_web_sm

  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/14.2 MB ? eta -:--:--
   ---- ----------------------------------- 1.6/14.2 MB 8.4 MB/s eta 0:00:02
   --------- ------------------------------ 3.4/14.2 MB 8.6 MB/s eta 0:00:02
   -------------- ------------------------- 5.2/14.2 MB 8.6 MB/s eta 0:00:02
   ------------------- -------------------- 7.1/14.2 MB 8.5 MB/s eta 0:00:01
   ------------------------ --------------- 8.7/14.2 MB 8.3 MB/s eta 0:00:01
   ------------------------------ --------- 10.7/14.2 MB 8.5 MB/s eta 0:00:01
   ----------------------------------- ---- 12.6/14.2 MB 8.5 MB/s eta 0:00:01
   ---------------------------------------  14.2/14.2 MB 8.5 MB/s eta 0:00:01
   ---------------------------------------- 14.2/14.2 MB 8.3 MB/s  0:00:01
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   -------------------


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ----- ---------------------------------- 1.8/12.8 MB 10.3 MB/s eta 0:00:02
     ------------- -------------------------- 4.2/12.8 MB 10.3 MB/s eta 0:00:01
     -------------------- ------------------- 6.6/12.8 MB 10.3 MB/s eta 0:00:01
     --------------------------- ------------ 8.7/12.8 MB 10.3 MB/s eta 0:00:01
     ------------------------------- ------- 10.5/12.8 MB 10.3 MB/s eta 0:00:01
     ------------------------------------ --- 11.8/12.8 MB 9.3 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 9.0 MB/s  0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import spacy  # analiza lenguaje natural
from IPython.display import Markdown, display #Mostrar la salida en formato markdown

# Cargar modelo spaCy 
nlp = spacy.load("en_core_web_sm")


def dicts_to_markdown_dossier_spacy(sections, main_title):
    """
    Convierte un dossier cinematográfico en Markdown a partir de diccionarios.
    
    sections: lista de diccionarios con 'titulo' y 'contenido'.
    main_title: título principal de la película.
    
    Se aplican keywords detectadas con spaCy, solo entidades nombradas.
    """
    md_lines = []

    def apply_keywords(text):
        doc = nlp(text)
        keywords = set()
        for ent in doc.ents:  # añade a keywords solo personas, lugares, organizaciones y misceláneos
            if ent.label_ in ["PERSON", "ORG", "GPE", "MISC"]:
                keywords.add(ent.text)
        # Aplicar negrita solo a las keywords de mas de 2 caracteres 
        for kw in keywords:
            if len(kw) > 2:
                text = text.replace(kw, f"**{kw}**")
        return text

    # Título principal
    md_lines.append(f"# {main_title}\n")

    # Secciones
    for section in sections:
        md_lines.append(f"## {section['titulo']}\n")
        md_lines.append(apply_keywords(section['contenido']) + "\n")

    return "\n".join(md_lines)


# EJEMPLO DE USO
dossier_sections = [
    {"titulo": "Informe Investigador",
     "contenido": """Matilda es una niña extremadamente inteligente.
Ama leer libros y tiene poderes telequinéticos.
Su familia no la comprende."""},
    
    {"titulo": "Análisis Narrativo",
     "contenido": """La historia sigue a Matilda mientras descubre sus habilidades y enfrenta a su directora estricta.
Su arco de personaje principal es superar la adversidad y encontrar un entorno seguro con Miss Honey."""},
    
    {"titulo": "Crítica",
     "contenido": """Una película entretenida con un buen balance entre humor y drama.
Los personajes son memorables y la dirección es sólida."""},
    
    {"titulo": "Recomendaciones",
     "contenido": """Recomendada para público infantil y juvenil.
Ideal para familias y amantes de historias de empoderamiento infantil."""}
]

main_title = "Matilda (1996)"

md_final = dicts_to_markdown_dossier_spacy(dossier_sections, main_title)

display(Markdown(md_final))


{'Matilda'}
{'Honey', 'un', 'arco de personaje', 'Matilda', 'la adversidad'}
{'un'}
{'de historias de empoderamiento', 'Recomendada', 'Ideal'}


# Matilda (1996)

## Informe Investigador

**Matilda** es una niña extremadamente inteligente.
Ama leer libros y tiene poderes telequinéticos.
Su familia no la comprende.

## Análisis Narrativo

La historia sigue a **Matilda** mientras descubre sus habilidades y enfrenta a su directora estricta.
Su **arco de personaje** principal es superar **la adversidad** y encontrar un entorno seguro con Miss **Honey**.

## Crítica

Una película entretenida con un buen balance entre humor y drama.
Los personajes son memorables y la dirección es sólida.

## Recomendaciones

**Recomendada** para público infantil y juvenil.
**Ideal** para familias y amantes **de historias de empoderamiento** infantil.


In [ ]:
# preguntar si lo puedo borrar(segun yo si)
import wikipedia 

def search_in_wikipedia(query: str, isMovie: bool = False):
    if isMovie:
        query += " (film)"

    try:
        summary = wikipedia.summary(query)
        url = wikipedia.page(query).url
        content = wikipedia.page(query).content

        return {"page_content": summary + content, "url": url}
    except wikipedia.exceptions.DisambiguationError as e:
        # Si hay ambigüedad, elegir la opción que contenga "film" o "película"
        for option in e.options:
            if "film" in option.lower() or "película" in option.lower():
                summary = wikipedia.summary(option)
                url = wikipedia.page(option).url
                content = wikipedia.page(query).content

                return {"page_content": summary + content, "url": url}
        return { "page_content": None, "url": f"No se encontró información en Wikipedia para '{query}'."}
    except Exception:
        return { "page_content": None, "url": f"No se encontró información en Wikipedia para '{query}'."}